Q1. How many lesson pages

In [2]:
# pull the lesson pages straight from the course repository. 

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader( # downloads the entire repository
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse() # returns dict with filname and content
    documents.append(doc)

In [4]:
files[0]

RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

In [5]:
# Q1. How many lessons pages

len(documents) # 72

72

Q2. Indexing and searching

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
documents[0]['filename'] # filename of the first index.

'01-agentic-rag/lessons/01-intro.md'

In [8]:
# Q2. What's the filename of the first result?
# 01-agentic-rag/lessons/14-agentic-loop.md

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3. RAG

In [9]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 15:08:53--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0.001s  

2026-06-22 15:08:53 (1.61 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [10]:
from rag_helper import RAGBase

class RAGBaseHomework(RAGBase):

    def search(self, query, num_results=5):
        boost_dict = {'content': 1.0}
        # filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            # filter_dict=filter_dict
        )


    def build_context(self, search_results): # this is content for the prompt.
        lines = [] # list

        for doc in search_results:
            #lines.append(f"F: {doc['filename']}")
            lines.append(f"A: {doc['content']}")
            lines.append("")  # spacing

        return "\n".join(lines).strip() # list to one string.
    

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response # .output_text    

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        print(response)
        answer = response.output[0].content[0].text
        usage = response.usage.input_tokens
        return answer, usage   

    

In [11]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBaseHomework(
    index=index,
    llm_client=openai_client,
)

answer, usage   = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f'{usage=}')
print(answer)


Response(id='resp_0d2af5fce079e32e006a394213fcf4819c94c557a44a46539b', created_at=1782137364.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0d2af5fce079e32e006a39421480a4819ca264b42c7fc6ac93', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model by checking whether the latest response contains any `function_call` items.\n\n- If there are function calls, the code runs the tool, appends the tool output to `messages`, and calls the model again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the key is this flag:\n\n```python\nhas_function_calls\n```\n\nand the exit condition:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: the model is called repeatedly inside a `while True` loop until it returns a response with no more tool calls.', type='output_text', logprobs=[])], role='assistant', sta

Q4. Chunking

In [12]:
# The lesson pages are long - some are thousands of characters. 
# Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. 
# A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).

In [13]:
len(chunks)
# 295

295

Q5. RAG with chunking

In [14]:
# Index the chunks

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [15]:
# using the chunks/not documents as earlier.

from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBaseHomework(
    index=index,
    llm_client=openai_client,
)

answer, usage   = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f'{usage=}')
print(answer)


Response(id='resp_0a7519f6c8d861c4006a39421a3868819c80f9c3af5f5fbe62', created_at=1782137370.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a7519f6c8d861c4006a39421b4148819ca82a127525bef1d6', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop. After each model response, the code checks whether any `function_call` items were returned.\n\n- If there is at least one function call, it runs the tool, appends the result to the messages, and loops again.\n- If there are no function calls in that turn, it `break`s out of the loop.\n\nSo the stop condition is: **no function calls in the model’s response**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, complete

In [16]:
# chunk usage=2231
# document usage=7046

# Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
# 3× fewer

7046/2231 # 3.15822

3.1582250112057375

Q6. Turning it into an agent

In [17]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> dict[str, str]: # ToyAIKit reads them and derives the schema for us.
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'content': 1.0}
    )



agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools() 

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [18]:
# Create the chat interface and a callback, then build the runner:

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [19]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

# Function call: search({"query":"agentic loop RAG difference"})
# Function call: search({"query":"agentic loop how it works"})
# Function call: search({"query":"plain RAG vs agentic loop"})

-> Response received


-> Response received
